# Lab | Data Aggregation and Filtering

This notebook cleans the marketing customer data and completes the aggregation, filtering, pivot-table, and melt exercises.

## Load and clean the data

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
data = pd.read_csv(url)
# Use consistent, easy-to-reference column names.
data.columns = (
    data.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
)
data = data.drop(columns=["unnamed:_0"])
data["effective_to_date"] = pd.to_datetime(
    data["effective_to_date"], format="%m/%d/%y"
)
data["month"] = data["effective_to_date"].dt.month
data["response_yes"] = data["response"].eq("Yes")

print("Dataset dimensions:", data.shape)
print("\nMissing values:\n", data.isna().sum().sort_values(ascending=False).head())

## 1. Low-claim customers who responded Yes

The new DataFrame contains customers with Total Claim Amount below 1,000 and a positive response to the campaign.

In [ ]:
low_claim_yes_customers = data.loc[
    (data["total_claim_amount"] < 1000)
    & (data["response"] == "Yes")
].copy()

print("Number of low-claim customers who responded Yes:", len(low_claim_yes_customers))
print(low_claim_yes_customers.head())

## 2. Average premium, CLV, and claims for Yes respondents

The grouped table compares policy type and gender. Customer Lifetime Value is used as the profitability indicator, while Total Claim Amount is used as a risk indicator.

In [ ]:
yes_responders = data.loc[data["response"] == "Yes"]

yes_segment_summary = (
    yes_responders.groupby(["policy_type", "gender"])
    .agg(
        average_monthly_premium=("monthly_premium_auto", "mean"),
        average_customer_lifetime_value=("customer_lifetime_value", "mean"),
        average_total_claim_amount=("total_claim_amount", "mean"),
        customers=("customer", "count")
    )
    .round(2)
    .sort_values("average_customer_lifetime_value", ascending=False)
)

print(yes_segment_summary)
print("\nMost profitable-looking Yes segment by average CLV:")
print(yes_segment_summary["average_customer_lifetime_value"].idxmax())
print("\nLowest average-claim Yes segment:")
print(yes_segment_summary["average_total_claim_amount"].idxmin())

A segment with high average CLV and relatively low average claims appears more attractive because it combines customer value with lower observed claim costs. The table should be read together with the customer count, since small segments can have unstable averages.

## 3. Customers by state

The first Series counts customers in every state. The second object keeps only states with more than 500 customers.

In [ ]:
customers_by_state = data["state"].value_counts()
states_over_500_customers = customers_by_state[customers_by_state > 500]

print("Customers by state:")
print(customers_by_state)
print("\nStates with more than 500 customers:")
print(states_over_500_customers)

## 4. Customer Lifetime Value by education and gender

The grouped table contains the maximum, minimum, and median CLV for every education-gender combination.

In [ ]:
clv_by_education_gender = (
    data.groupby(["education", "gender"])["customer_lifetime_value"]
    .agg(maximum="max", minimum="min", median="median")
    .round(2)
)

print(clv_by_education_gender)
print("\nCombination with the highest median CLV:")
print(clv_by_education_gender["median"].idxmax())
print(clv_by_education_gender["median"].max())

The median is the most useful comparison for a typical customer because it is less affected by extreme CLV values. The combination with the highest median CLV represents the strongest typical customer-value segment.

## Bonus 5. Policies sold by state and month

Number of Policies is summed because it records how many policies each customer holds.

In [ ]:
policies_by_state_month = pd.pivot_table(
    data,
    index="state",
    columns="month",
    values="number_of_policies",
    aggfunc="sum",
    fill_value=0
).sort_index()

print(policies_by_state_month)

## Bonus 6. Monthly policies for the top three states

The top states are selected using total policies sold across all months, then the monthly values are displayed in a new DataFrame.

In [ ]:
state_policy_totals = (
    data.groupby("state")["number_of_policies"]
    .sum()
    .sort_values(ascending=False)
)
top_3_states = state_policy_totals.head(3).index.tolist()

top_3_state_monthly_policies = policies_by_state_month.loc[top_3_states]

print("Top three states:", top_3_states)
print(top_3_state_monthly_policies)

## Bonus 7. Response rate by marketing channel

The channel column is converted into indicator columns, melted into long format, and then aggregated to calculate the percentage of customers responding Yes in each channel.

In [ ]:
channel_indicators = pd.get_dummies(data["sales_channel"], dtype=int)
channel_indicators["response_yes"] = data["response_yes"].astype(int).values

melted_channels = channel_indicators.melt(
    id_vars="response_yes",
    var_name="marketing_channel",
    value_name="customer_in_channel"
)
melted_channels = melted_channels.loc[
    melted_channels["customer_in_channel"] == 1
]

response_rate_by_channel = (
    melted_channels.groupby("marketing_channel")
    .agg(
        customers=("customer_in_channel", "size"),
        responders=("response_yes", "sum")
    )
)
response_rate_by_channel["response_rate"] = (
    response_rate_by_channel["responders"]
    / response_rate_by_channel["customers"]
    * 100
).round(2)
response_rate_by_channel = response_rate_by_channel.sort_values(
    "response_rate", ascending=False
)

print(response_rate_by_channel)